In [3]:
import os
import glob
import joblib
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc,
    matthews_corrcoef
)

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
PUBCHEM_FILE = "dataset/pubchem_clean.csv"
MODEL_DIR = "saved_models/chembl_morgan"
OUTPUT_FILE = "pubchem_test/metrics_morgan.csv"

# ------------------------------------------------------------
# Load dataset
# ------------------------------------------------------------
df = pd.read_csv(PUBCHEM_FILE)

smiles_col = "standard_smiles"
label_col = "class"

if smiles_col not in df.columns:
    raise ValueError(f"Column '{smiles_col}' not found in {PUBCHEM_FILE}")

if label_col not in df.columns:
    raise ValueError(
        f"Column '{label_col}' not found in {PUBCHEM_FILE}. "
        f"You need true labels to compute the evaluation table."
    )

# ------------------------------------------------------------
# Convert SMILES to molecules
# ------------------------------------------------------------
df["mol"] = df[smiles_col].apply(Chem.MolFromSmiles)
df = df[df["mol"].notnull()].copy().reset_index(drop=True)

print(f"Valid molecules: {len(df)}")

# ------------------------------------------------------------
# Generate the SAME fingerprint used during training
# ------------------------------------------------------------
# ✅ Generate Morgan Keys
def mol_to_morgan_counts(mol, radius=2, nBits=2048):
    fp = rdMolDescriptors.GetHashedMorganFingerprint(
        mol,
        radius=radius,
        nBits=nBits
    )
    # Convert to dense NumPy array
    arr = np.zeros((nBits,), dtype=int)
    for idx, value in fp.GetNonzeroElements().items():
        arr[idx] = value
    return arr

df['fp'] = df['mol'].apply(mol_to_morgan_counts)

# Convert fingerprints to NumPy matrix
X = np.vstack(df['fp'].values)
y = df['class'].astype(int)


# ------------------------------------------------------------
# Metrics functions
# ------------------------------------------------------------
def enrichment_factor(y_true, y_scores, top_percent=0.01):
    sorted_idx = np.argsort(y_scores)[::-1]
    y_true_sorted = np.array(y_true)[sorted_idx]

    n_total = len(y_true)
    n_actives = np.sum(y_true)

    top_n = int(np.ceil(top_percent * n_total))
    top_hits = y_true_sorted[:top_n]

    n_top_actives = np.sum(top_hits)

    if n_actives == 0 or top_n == 0:
        return 0.0

    ef = (n_top_actives / top_n) / (n_actives / n_total)
    return ef


def bedroc_score(y_true, y_scores, alpha=20.0):
    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    order = np.argsort(y_scores)[::-1]
    y_true = y_true[order]

    n = len(y_true)
    n_pos = np.sum(y_true)

    if n_pos == 0:
        return 0.0

    ranks = np.where(y_true == 1)[0] + 1   # 1-based ranks
    ri = ranks / n

    rie = np.sum(np.exp(-alpha * ri)) / n_pos

    rand = (1 - np.exp(-alpha)) / alpha
    max_rie = (1 - np.exp(-alpha * (n_pos / n))) / (alpha * (n_pos / n))

    if max_rie == rand:
        return 0.0

    bedroc = (rie - rand) / (max_rie - rand)
    return bedroc

# ------------------------------------------------------------
# Load all saved models
# ------------------------------------------------------------
model_files = sorted(glob.glob(os.path.join(MODEL_DIR, "*.joblib")))

if not model_files:
    raise FileNotFoundError(f"No saved model found in {MODEL_DIR}")

# nice model names from filenames
def clean_model_name(path):
    name = os.path.splitext(os.path.basename(path))[0]
    return name.replace("_", " ").title()

# ------------------------------------------------------------
# Evaluate all models on PubChem
# ------------------------------------------------------------
results = []

for model_path in model_files:
    model_name = clean_model_name(model_path)
    print(f"\nEvaluating {model_name} ...")

    model = joblib.load(model_path)

    y_pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X)[:, 1]
    else:
        y_prob = None

    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    f1 = f1_score(y, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y, y_pred)

    if y_prob is not None:
        roc = roc_auc_score(y, y_prob)

        p, r, _ = precision_recall_curve(y, y_prob)
        pra = auc(r, p)

        ef1 = enrichment_factor(y, y_prob, top_percent=0.01)
        ef5 = enrichment_factor(y, y_prob, top_percent=0.05)
        ef10 = enrichment_factor(y, y_prob, top_percent=0.10)

        bedroc = bedroc_score(y, y_prob, alpha=20)
    else:
        roc = np.nan
        pra = np.nan
        ef1 = np.nan
        ef5 = np.nan
        ef10 = np.nan
        bedroc = np.nan

    results.append([
        model_name, acc, prec, rec, f1, roc, pra, ef1, ef5, ef10, bedroc, mcc
    ])

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------
columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "EF@1%",
    "EF@5%",
    "EF@10%",
    "BEDROC",
    "MCC"
]

results_df = pd.DataFrame(results, columns=columns)
results_df = results_df.sort_values("PR_AUC", ascending=False)

print("\n==================== PUBCHEM TEST RESULTS ====================")
print(results_df)

results_df.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved results to: {OUTPUT_FILE}")

Valid molecules: 689

Evaluating Adaboost ...

Evaluating Decision Tree ...

Evaluating Extra Trees ...

Evaluating Gradient Boosting ...

Evaluating Knn ...

Evaluating Lightgbm ...

Evaluating Logistic Regression ...

Evaluating Naive Bayes ...

Evaluating Random Forest ...

Evaluating Svm ...

==================== PUBCHEM TEST RESULTS ====================
                 Model  Accuracy  Precision    Recall        F1   ROC_AUC  \
8        Random Forest  0.791001   0.641935  0.857759  0.734317  0.898514   
2          Extra Trees  0.764877   0.611465  0.827586  0.703297  0.886408   
9                  Svm  0.802612   0.656863  0.866379  0.747212  0.895679   
3    Gradient Boosting  0.756168   0.593023  0.879310  0.708333  0.881909   
5             Lightgbm  0.766328   0.611285  0.840517  0.707804  0.874076   
6  Logistic Regression  0.670537   0.506562  0.831897  0.629690  0.832047   
4                  Knn  0.711176   0.547009  0.827586  0.658662  0.806445   
1        Decision Tree 